eda：https://www.kaggle.com/code/waterjoe/birdclef2026-eda

train code：https://www.kaggle.com/code/waterjoe/birdclef2026-train-google-perch

thanks:[Antoine Masq 0.792](https://www.kaggle.com/code/antoinemasq/birdclef-2026-pytorch-baseline-training)

In [ ]:
# ==============================
# 1. 环境配置 (纯 CPU 模式，允许 XLA 编译)
# ==============================
import os
import warnings

# 彻底对当前进程屏蔽 GPU，Kaggle 环境下强制走 CPU
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'   # 屏蔽烦人的 info 日志

os.environ['JAX_PLATFORMS'] = 'cpu'

warnings.filterwarnings('ignore')

import jax # 👈 新增：提前唤醒 JAX 的 CPU 后端
jax.config.update('jax_platform_name', 'cpu') # 👈 新增

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import tensorflow as tf

# 确保 TensorFlow 完全不初始化 GPU 显存
tf.config.set_visible_devices([], 'GPU')

# ==============================
# 2. 基础配置与路径
# ==============================
ROOT = "/kaggle/input/competitions/birdclef-2026"
TEST_DIR = os.path.join(ROOT, "test_soundscapes")
TRAIN_DIR = os.path.join(ROOT, "train_soundscapes") 

# 使用你指定的 CPU 版 Perch 模型
PERCH_PATH = '/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1'
#/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2/2
PT_MODEL_PATH = '/kaggle/input/notebooks/waterjoe/birdclef2026-train-google-perch/classifier_head.pth' 

SR = 32000
WIN_SEC = 5
CHUNK_SAMPLES = SR * WIN_SEC       # 160,000
TOTAL_SAMPLES = CHUNK_SAMPLES * 12 # 1,920,000 (60秒)
NUM_CLASSES = 234
EMBEDDING_DIM = 1536

# 强制 PyTorch 使用 CPU
device = torch.device("cpu")
sample_sub = pd.read_csv(os.path.join(ROOT, "sample_submission.csv"))
CLASS_LABELS = sample_sub.columns[1:].tolist()

print(f"Classes: {len(CLASS_LABELS)} | Operating on strictly: {device.type.upper()}")

# ==============================
# 3. 加载双模型 (CPU)
# ==============================
print("Loading TF Perch Model on CPU...")
perch = tf.saved_model.load(PERCH_PATH)
infer_fn = perch.signatures['serving_default']

print("Loading PT Classifier Head on CPU...")
class ClassifierHead(nn.Module):
    def __init__(self, input_dim=1536, num_classes=234):
        super().__init__()
        self.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(input_dim, 512),
            nn.GELU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.head(x)

model = ClassifierHead(input_dim=EMBEDDING_DIM, num_classes=NUM_CLASSES)
if os.path.exists(PT_MODEL_PATH):
    # 确保 map_location 指向 cpu
    model.load_state_dict(torch.load(PT_MODEL_PATH, map_location=device))
    print("✓ PyTorch weights loaded successfully.")
else:
    print("⚠️ PyTorch weights NOT FOUND! Using random initialization.")
    
model.to(device)
model.eval()

# ==============================
# 4. 辅助函数
# ==============================
def temporal_smooth(pred):
    """简单的时间平滑，利用前后时间窗的上下文"""
    smoothed = pred.copy()
    for i in range(len(pred)):
        p0 = pred[i]
        p_prev = pred[max(i-1, 0)]
        p_next = pred[min(i+1, len(pred)-1)]
        smoothed[i] = 0.6 * p0 + 0.2 * p_prev + 0.2 * p_next
    return smoothed

def fast_load_audio(path):
    """后台多线程使用该函数快速读取音频"""
    audio, _ = sf.read(path)
    # soundfile 读取有时是 float64，这里统一转为 float32
    audio = audio.astype(np.float32) 
    
    if len(audio) < TOTAL_SAMPLES:
        audio = np.pad(audio, (0, TOTAL_SAMPLES - len(audio)))
    else:
        audio = audio[:TOTAL_SAMPLES]
        
    # 直接提前 reshape 好，减轻主线程负担
    return audio.reshape(12, CHUNK_SAMPLES)

def run_inference(audio_batch):
    """主线程专门负责跑纯数学计算"""
    # 1. TF Perch 提取特征
    tf_tensor = tf.convert_to_tensor(audio_batch, dtype=tf.float32)
    embeddings = infer_fn(inputs=tf_tensor)['embedding'].numpy()
            
    # 2. PyTorch Head 进行分类
    pt_tensor = torch.from_numpy(embeddings).to(device)
    with torch.no_grad():
        logits = model(pt_tensor)
        probs = torch.sigmoid(logits).numpy() # 直接在 CPU 上转为 numpy
        
    return temporal_smooth(probs)

# ==============================
# 5. 推理流水线 (多线程预加载 + 单线程推理)
# ==============================
files = sorted([f for f in os.listdir(TEST_DIR) if f.endswith('.ogg')])

if len(files) == 0:
    print(f"⚠️ {TEST_DIR} 为空，切换到 train_soundscapes 进行测试...")
    if os.path.exists(TRAIN_DIR):
        files = sorted([f for f in os.listdir(TRAIN_DIR) if f.endswith('.ogg')])[:2]
        TEST_DIR = TRAIN_DIR
    else:
        pd.DataFrame(columns=['row_id'] + CLASS_LABELS).to_csv("submission.csv", index=False)
        print("Empty submission generated.")
        exit()

print(f"📁 Processing {len(files)} files via CPU Pipeline...")

all_rows = []
all_preds = []

# 使用 2-4 个线程做后台 I/O 足矣，留出大部分 CPU 核心给 TF/PT 做矩阵运算
WORKERS = min(4, os.cpu_count() or 1) 

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    # 按照 files 的顺序提交任务并获取 Future 对象
    future_to_file = {executor.submit(fast_load_audio, os.path.join(TEST_DIR, f)): f for f in files}
    
    # 注意这里不使用 as_completed，而是按照原列表 files 顺序取结果，保证 row_id 顺序不出错
    for f in tqdm(files):
        # 找到对应这个文件的 future 并获取结果 (此时后台多半已经把这个音频读好了)
        future = [k for k, v in future_to_file.items() if v == f][0]
        audio_batch = future.result() 
        
        # 执行推理
        file_preds = run_inference(audio_batch)
        
        # 保存结果
        base_name = f.replace(".ogg", "")
        for i in range(12):
            seconds = (i + 1) * 5
            all_rows.append(f"{base_name}_{seconds}")
            all_preds.append(file_preds[i])

# ==============================
# 6. 生成 Submission
# ==============================
if len(all_preds) > 0:
    submission = pd.DataFrame(all_preds, columns=CLASS_LABELS)
    submission.insert(0, "row_id", all_rows)
else:
    submission = pd.DataFrame(columns=['row_id'] + CLASS_LABELS)

submission.to_csv("submission.csv", index=False)
print(f"✅ Submission saved! Shape: {submission.shape}")

In [ ]:
%%writefile lb792.py

from warnings import filterwarnings
filterwarnings("ignore")

import os
import glob
import timm
import torch
import torch.nn as nn
import torchaudio
import torchvision
import numpy as np
import pandas as pd
import openvino as ov
from tqdm import tqdm
from torch.utils.data import Dataset

ov_device       = "CPU"
PATH            = '/kaggle/input/competitions/birdclef-2026/'
TEST_PATH       = PATH + 'test_soundscapes/'
TRAIN_PATH      = PATH + 'train_soundscapes/'
N_FALLBACK      = 2
SUBMISSION_FILE = "lb792.csv"
model_dir       = "/kaggle/input/notebooks/antoinemasq/birdclef-2026-pytorch-baseline-training/models"

DUR = 5
SR  = 32000

class Spectrogram(nn.Module):
    def __init__(self, sr=32000, n_fft=2048, n_mels=256, hop_length=512,
                 f_min=20, f_max=16000, channels=1, norm="slaney",
                 mel_scale="htk", target_size=(256, 256), top_db=80.0, **kwargs):
        super().__init__()
        self.channels = channels
        self.top_db   = top_db
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr, n_fft=n_fft, hop_length=hop_length,
            n_mels=n_mels, f_min=f_min, f_max=f_max,
            mel_scale=mel_scale, pad_mode="reflect", power=2.0,
            norm=norm, center=True,
        )
        self.resize = torchvision.transforms.Resize(size=target_size)

    def power_to_db(self, S):
        amin     = 1e-10
        log_spec = 10.0 * torch.log10(S.clamp(min=amin))
        log_spec -= 10.0 * torch.log10(torch.tensor(amin).to(S))
        if self.top_db is not None:
            max_val  = log_spec.flatten(-2).max(dim=-1).values[..., None, None]
            log_spec = torch.maximum(log_spec, max_val - self.top_db)
        return log_spec

    def forward(self, x, resize=True):
        squeeze = x.dim() == 1
        if squeeze:
            x = x.unsqueeze(0)
        mel = self.mel_transform(x)
        mel = self.power_to_db(mel)
        mel = mel.unsqueeze(1).repeat(1, self.channels, 1, 1)
        if resize:
            mel = self.resize(mel)
        B, C = mel.shape[:2]
        flat = mel.view(B, C, -1)
        mins = flat.min(dim=-1).values[..., None, None]
        maxs = flat.max(dim=-1).values[..., None, None]
        mel  = (mel - mins) / (maxs - mins + 1e-7)
        if squeeze:
            mel = mel.squeeze(0)
        return mel

class BirdModel(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        cfg = {
            'backbone':        'tf_efficientnetv2_b0',
            'backbone_pooling':'avg',
            'dropout':          0.1,
            'pretrained':       False,
            'channels':         1,
            'num_labels':       234, # Will dynamically update
        }
        if config:
            cfg.update(config)
        self.backbone = timm.create_model(
            cfg['backbone'],
            pretrained=cfg['pretrained'],
            num_classes=cfg['num_labels'],
            global_pool=cfg['backbone_pooling'],
            in_chans=cfg['channels'],
            drop_rate=cfg['dropout'],
        )

    def forward(self, x):
        return self.backbone(x)


class BirdDataset(Dataset):
    def __init__(self, paths, spec_transform):
        self.paths = paths
        self.spec  = spec_transform
        self.chunk_len = SR * DUR
        self.half_chunk = self.chunk_len // 2

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        filepath = self.paths[idx]
        filename = filepath.split('/')[-1].split('.')[0]
        try:
            wav, _ = torchaudio.load(filepath)
            wav = wav.float()[0]  
            
            n_seg = len(wav) // self.chunk_len 
            
            if n_seg == 0:
                wav = torch.nn.functional.pad(wav, (0, self.chunk_len - len(wav)))
                n_seg = 1
            else:
                wav = wav[: n_seg * self.chunk_len]
            
            # 1. Standard Chunks (0-5s, 5-10s, ...)
            wav_std = wav.reshape((n_seg, self.chunk_len))
            
            # 2. Shifted Chunks for TTA (2.5-7.5s, 7.5-12.5s, ...)
            wav_padded = torch.nn.functional.pad(wav, (0, self.half_chunk))
            wav_shift = torch.stack([
                wav_padded[i * self.chunk_len + self.half_chunk : (i+1) * self.chunk_len + self.half_chunk]
                for i in range(n_seg)
            ])
            
            mel_std   = torch.stack([self.spec(wav_std[i]) for i in range(n_seg)])
            mel_shift = torch.stack([self.spec(wav_shift[i]) for i in range(n_seg)])
            names =[f"{filename}_{int((i + 1) * DUR)}" for i in range(n_seg)]
            
            return mel_std.numpy().astype(np.float32), mel_shift.numpy().astype(np.float32), names
            
        except Exception as e:
            print(f"Error loading {filepath}: {e}")
            n_seg = int(240 / DUR) 
            mel_std   = torch.zeros((n_seg, 1, 256, 256))
            mel_shift = torch.zeros((n_seg, 1, 256, 256))
            names =[f"{filename}_{int((i + 1) * DUR)}" for i in range(n_seg)]
            return mel_std.numpy().astype(np.float32), mel_shift.numpy().astype(np.float32), names

def export_to_openvino(ckpt_path, ov_model_path, config=None):
    if os.path.exists(ov_model_path):
        return
    state = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    
    num_labels = state['backbone.classifier.weight'].shape[0]
    cfg = config.copy() if config else {}
    cfg['num_labels'] = num_labels
    
    model = BirdModel(cfg)
    model.load_state_dict(state)
    model.eval()
    
    dummy    = torch.zeros(1, 1, 256, 256)
    ov_model = ov.convert_model(model, example_input=dummy)
    ov_model.reshape([-1, 1, 256, 256]) # Support dynamic batch sizes
    ov.save_model(ov_model, ov_model_path)

def compile_ov_model(ov_model_path):
    core     = ov.Core()
    ov_model = core.read_model(ov_model_path)
    return core.compile_model(ov_model, device_name=ov_device)

def predict_batch(compiled_models, batch_np):
    logits_list =[]
    for compiled in compiled_models:
        logits = compiled(batch_np)[compiled.output(0)]
        logits_list.append(logits)
    
    # 1. Ensemble average the logits directly
    mean_logits = np.mean(logits_list, axis=0)
    
    # 2. Standard sigmoid (removed temperature scaling to preserve native calibration)
    probs = 1.0 / (1.0 + np.exp(np.clip(-mean_logits, -50, 50)))
    return probs

def run_inference(compiled_models, paths, spec):
    dataset  = BirdDataset(paths, spec)
    all_preds, all_names = [],[]

    for mel_std, mel_shift, names in tqdm(dataset, desc="Infer"):
        
        preds_std   = predict_batch(compiled_models, mel_std)
        preds_shift = predict_batch(compiled_models, mel_shift)
        
        # 1. Weave TTA Shifted Predictions (Recovers birds cut in half)
        preds = np.copy(preds_std)
        n = len(preds)
        for i in range(n):
            shift_prev = preds_shift[i-1] if i > 0 else preds_std[i]
            shift_curr = preds_shift[i]
            # 50% Standard Window + 50% overlapping halves
            preds[i] = 0.50 * preds_std[i] + 0.25 * shift_prev + 0.25 * shift_curr
        
        # 2. Wider Gaussian Time-Domain Smoothing
        smoothed_preds = np.copy(preds)
        for i in range(n):
            p_m2 = preds[max(0, i-2)]
            p_m1 = preds[max(0, i-1)]
            p_0  = preds[i]
            p_p1 = preds[min(n-1, i+1)]
            p_p2 = preds[min(n-1, i+2)]
            # Smooth the probabilities out across a 25-second window
            smoothed_preds[i] = 0.05*p_m2 + 0.15*p_m1 + 0.60*p_0 + 0.15*p_p1 + 0.05*p_p2
            
        # 3. Global File Context (The Grandmaster Trick, without clipping!)
        file_max = np.max(smoothed_preds, axis=0)
        
        # Blend: 80% local chunk + 20% global file context
        # This securely boosts the AUC rank of chunks in files where the bird is confirmed to exist
        final_preds = 0.80 * smoothed_preds + 0.20 * file_max
        
        all_preds.append(final_preds)
        all_names.extend(names)

    return np.concatenate(all_preds, axis=0), all_names


def create_submission(preds, names, label_cols):
    sample_sub = pd.read_csv(PATH + 'sample_submission.csv')
    expected_cols = sample_sub.columns.tolist()
    
    if len(preds) == 0:
        sample_sub.to_csv(SUBMISSION_FILE, index=False)
        return sample_sub
        
    df_preds = pd.DataFrame(preds, columns=label_cols[:preds.shape[1]])
    df_preds['row_id'] = names
    
    final_dict = {'row_id': df_preds['row_id']}
    for col in expected_cols:
        if col == 'row_id': continue
        if col in df_preds.columns:
            final_dict[col] = df_preds[col].values
        else:
            final_dict[col] = 0.0
            
    final_df = pd.DataFrame(final_dict)
    final_df = final_df[expected_cols]
    final_df.to_csv(SUBMISSION_FILE, index=False)
    
    print(f"Submission saved: {final_df.shape}\n\n")
    return final_df

def main():
    sample_sub = pd.read_csv(PATH + 'sample_submission.csv')
    label_cols = [c for c in sample_sub.columns if c != 'row_id']

    # 1. Search for test files (Using glob to support recursive search)
    paths = glob.glob(TEST_PATH + "**/*.ogg", recursive=True)
    
    if not paths:
        paths = sorted(glob.glob(TRAIN_PATH + "**/*.ogg", recursive=True))[:N_FALLBACK]

    # 3. Handle absolutely no files case elegantly!
    if not paths:
        print(f"Files: 0. Generating default submission directly to avoid crashes.")
        create_submission([], [], label_cols)
        return

    print(f"Files: {len(paths)}, Model Output Classes: {len(label_cols)}")

    spec = Spectrogram(sr=SR, n_fft=2048, n_mels=256, hop_length=512,
                       f_min=20, f_max=16000, channels=1,
                       target_size=(256, 256), top_db=80.0)

    all_ckpts =[os.path.join(model_dir, f) for f in sorted(os.listdir(model_dir)) if f.endswith('.pth')]
    best_ckpts =[p for p in all_ckpts if '_best.pth' in os.path.basename(p)]
    ckpt_paths = best_ckpts if best_ckpts else all_ckpts

    ov_export_dir = "/kaggle/working/ov_models"
    os.makedirs(ov_export_dir, exist_ok=True)

    compiled_models =[]
    for ckpt in ckpt_paths:
        ov_name = os.path.splitext(os.path.basename(ckpt))[0] + '.xml'
        ov_path = os.path.join(ov_export_dir, ov_name)
        export_to_openvino(ckpt, ov_path)
        compiled_models.append(compile_ov_model(ov_path))

    print(f"Loaded {len(compiled_models)} OpenVINO model(s)")

    preds, names = run_inference(compiled_models, paths, spec)
    create_submission(preds, names, label_cols)

if __name__ == "__main__":
    main()

In [ ]:
%%writefile blend.py
import pandas as pd, os
from warnings import filterwarnings
filterwarnings("ignore")

print(f"\n---> Starting model blend")

sub825 = pd.read_csv(f"submission.csv", index_col = "row_id")
sub792 = pd.read_csv(f"lb792.csv", index_col = "row_id")[sub825.columns]

sub_fl = sub792 * 0.4 + sub825 * 0.6

print(f"---> Blended submission file shape = {sub_fl.shape}\n")
sub_fl.to_csv(f"submission.csv", index = True)

In [ ]:
import pandas as pd, os, time, sys

# !python lb825.py
!python lb792.py
!python blend.py

print()
display(
    pd.read_csv(
        "submission.csv", index_col = "row_id"
    ).head(5)
)

print(f"✅ Submission saved! Shape: {submission.shape}")
print()
!ls

In [ ]:
# ###GPU over time

# import os

# import warnings

# import numpy as np

# import pandas as pd

# import librosa

# import torch

# import torch.nn as nn

# from tqdm import tqdm



# # ==============================

# # 1. TF 环境配置 (防止双框架显存打架)

# # ==============================

# os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'

# os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

# os.environ['TF_XLA_AUTO_JIT'] = '0' 



# warnings.filterwarnings('ignore')



# import tensorflow as tf

# tf.config.optimizer.set_jit(False) 



# # 核心：必须开启显存按需增长，给 PyTorch 留足空间

# gpus = tf.config.list_physical_devices('GPU')

# if gpus:

#     try:

#         for gpu in gpus:

#             tf.config.experimental.set_memory_growth(gpu, True)

#     except RuntimeError as e:

#         print(f"GPU config error: {e}")



# # ==============================

# # 2. 基础配置与路径

# # ==============================

# ROOT = "/kaggle/input/competitions/birdclef-2026"

# TEST_DIR = os.path.join(ROOT, "test_soundscapes")

# TRAIN_DIR = os.path.join(ROOT, "train_soundscapes") 



# PERCH_PATH = '/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1'

# PT_MODEL_PATH = '/kaggle/input/notebooks/waterjoe/birdclef2026-train-google-perch/classifier_head.pth' 



# SR = 32000

# WIN_SEC = 5

# CHUNK_SAMPLES = SR * WIN_SEC       # 160,000

# TOTAL_SAMPLES = CHUNK_SAMPLES * 12 # 1,920,000 (60秒)

# NUM_CLASSES = 234

# EMBEDDING_DIM = 1536



# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# sample_sub = pd.read_csv(os.path.join(ROOT, "sample_submission.csv"))

# CLASS_LABELS = sample_sub.columns[1:].tolist()



# print(f"Classes: {len(CLASS_LABELS)} | PyTorch Device: {device}")



# # ==============================

# # 3. 加载双模型 (全部放入 GPU)

# # ==============================

# print("Loading TF Perch Model on GPU...")

# # ✅ 修正：直接加载，让 TF 自动把计算图放在 GPU 上

# perch = tf.saved_model.load(PERCH_PATH)

# infer_fn = perch.signatures['serving_default']



# print("Loading PT Classifier Head on GPU...")

# class ClassifierHead(nn.Module):

#     def __init__(self, input_dim=1536, num_classes=234):

#         super().__init__()

#         self.head = nn.Sequential(

#             nn.Dropout(0.4),

#             nn.Linear(input_dim, 512),

#             nn.GELU(),

#             nn.BatchNorm1d(512),

#             nn.Dropout(0.3),

#             nn.Linear(512, 256),

#             nn.GELU(),

#             nn.BatchNorm1d(256),

#             nn.Dropout(0.2),

#             nn.Linear(256, num_classes)

#         )

#     def forward(self, x):

#         return self.head(x)



# model = ClassifierHead(input_dim=EMBEDDING_DIM, num_classes=NUM_CLASSES)

# if os.path.exists(PT_MODEL_PATH):

#     model.load_state_dict(torch.load(PT_MODEL_PATH, map_location=device))

#     print("✓ PyTorch weights loaded successfully.")

# else:

#     print("⚠️ PyTorch weights NOT FOUND! Using random initialization.")

    

# model.to(device)

# model.eval()



# # ==============================

# # 4. 时间平滑函数

# # ==============================

# def temporal_smooth(pred):

#     smoothed = pred.copy()

#     for i in range(len(pred)):

#         p0 = pred[i]

#         p_prev = pred[max(i-1, 0)]

#         p_next = pred[min(i+1, len(pred)-1)]

#         smoothed[i] = 0.6 * p0 + 0.2 * p_prev + 0.2 * p_next

#     return smoothed



# # ==============================

# # 5. 预测核心逻辑 

# # ==============================

# def predict_file(path):

#     audio, _ = librosa.load(path, sr=SR)

    

#     if len(audio) < TOTAL_SAMPLES:

#         audio = np.pad(audio, (0, TOTAL_SAMPLES - len(audio)))

#     else:

#         audio = audio[:TOTAL_SAMPLES]

        

#     audio_batch = audio.reshape(12, CHUNK_SAMPLES).astype(np.float32)

#     tf_tensor = tf.convert_to_tensor(audio_batch, dtype=tf.float32)

    

#     # ✅ 修正：直接执行，TF 会在 GPU 上调用 XLA 算子

#     embeddings = infer_fn(inputs=tf_tensor)['embedding'].numpy()

            

#     # ✅ 修正：使用 from_numpy 避免内存拷贝，直接送入 device

#     pt_tensor = torch.from_numpy(embeddings).to(device)

#     with torch.no_grad():

#         logits = model(pt_tensor)

#         probs = torch.sigmoid(logits).cpu().numpy() 

        

#     probs = temporal_smooth(probs)

#     return probs



# # ==============================

# # 6. 推理循环 & 7. 生成 Submission

# # ==============================

# files = sorted([f for f in os.listdir(TEST_DIR) if f.endswith('.ogg')])



# if len(files) == 0:

#     print(f"⚠️ {TEST_DIR} 为空，切换到 train_soundscapes 进行测试...")

#     if os.path.exists(TRAIN_DIR):

#         files = sorted([f for f in os.listdir(TRAIN_DIR) if f.endswith('.ogg')])[:2]

#         TEST_DIR = TRAIN_DIR

#     else:

#         pd.DataFrame(columns=['row_id'] + CLASS_LABELS).to_csv("submission.csv", index=False)

#         print("Empty submission generated.")

#         exit()



# print(f"📁 Processing {len(files)} files...")



# all_rows = []

# all_preds = []



# for f in tqdm(files):

#     path = os.path.join(TEST_DIR, f)

#     file_preds = predict_file(path) 

#     base_name = f.replace(".ogg", "")

    

#     for i in range(12):

#         seconds = (i + 1) * 5

#         all_rows.append(f"{base_name}_{seconds}")

#         all_preds.append(file_preds[i])



# if len(all_preds) > 0:

#     submission = pd.DataFrame(all_preds, columns=CLASS_LABELS)

#     submission.insert(0, "row_id", all_rows)

# else:

#     submission = pd.DataFrame(columns=['row_id'] + CLASS_LABELS)



# submission.to_csv("submission.csv", index=False)

# print(f"✅ Submission saved! Shape: {submission.shape}")